# Tarea 4 y 4 – Análisis con MLlib en PySpark

**Materia:** Datos Masivos  
**Alumno:** Sergio Cortes Cepeda  

## Fuente de datos

Los datos utilizados provienen del repositorio Zenodo:  
https://zenodo.org/records/7923702

Se trabajó con los meses de enero, febrero y marzo de los años 2019 a 2022.

## Introducción

En esta tarea realice un análisis de vuelos utilizando PySpark y la librería MLlib.  
Debido al gran volumen de información, los archivos fueron cargados de manera segmentada por año y por mes, para posteriormente unificarse en un solo DataFrame.

El objetivo del trabajo es construir un modelo de regresión capaz de estimar la duración de un vuelo a partir de variables como el aeropuerto de origen, destino, tipo de aeronave y coordenadas geográficas.

## Comentario

En esta parte tuve que instalar otra version de pyspark, debido a que mi librerias no me permitian correr mi trabajo

In [4]:
!pip uninstall pyspark -y
!pip install pyspark==3.5.0

Found existing installation: pyspark 4.1.1
Uninstalling pyspark-4.1.1:
  Successfully uninstalled pyspark-4.1.1
     ---------------------------------------- 0.0/316.9 MB ? eta -:--:--
     --------------------------------------- 1.3/316.9 MB 11.2 MB/s eta 0:00:29
      -------------------------------------- 4.5/316.9 MB 13.4 MB/s eta 0:00:24
      -------------------------------------- 7.3/316.9 MB 13.7 MB/s eta 0:00:23
     - ------------------------------------ 10.2/316.9 MB 13.9 MB/s eta 0:00:23
     - ------------------------------------ 12.1/316.9 MB 12.6 MB/s eta 0:00:25
     - ------------------------------------ 14.7/316.9 MB 12.6 MB/s eta 0:00:24
     -- ----------------------------------- 17.6/316.9 MB 12.7 MB/s eta 0:00:24
     -- ----------------------------------- 20.7/316.9 MB 13.0 MB/s eta 0:00:23
     -- ----------------------------------- 23.6/316.9 MB 13.0 MB/s eta 0:00:23
     --- ---------------------------------- 26.5/316.9 MB 13.1 MB/s eta 0:00:23
     --- ------

In [1]:
import pyspark
print(pyspark.__version__)

3.5.0


In [3]:
!pip install numpy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 11.2 MB/s eta 0:00:02
   -------------- ------------------------- 4.7/12.6 MB 15.0 MB/s eta 0:00:01
   ------------------------ --------------- 7.6/12.6 MB 14.2 MB/s eta 0:00:01
   --------------------------------- ------ 10.5/12.6 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 13.4 MB/s  0:00:01


In [1]:
import numpy as np
print(np.__version__)

2.4.4


In [3]:
##Creacion de sesion en sapark 

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, unix_timestamp, round
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

spark = SparkSession.builder \
    .appName("Tarea4_5_Vuelos_MLlib") \
    .getOrCreate()

## Carga de datos

Debido al tamaño de los archivos, la carga de información se realizó por mes y por año.  
Se deshabilitó la inferencia automática del esquema (`inferSchema=False`) para optimizar el rendimiento en la lectura de los archivos.

Posteriormente, todos los DataFrames fueron unidos en uno solo para facilitar su análisis y modelado.

In [4]:
df_2019_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Enero.csv")

df_2019_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Febrero.csv")

df_2019_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2019/flightlist_2019Marzo.csv")

df_2019 = df_2019_ene.union(df_2019_feb).union(df_2019_mar)

In [5]:
df_2020_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2020/flightlist_2020Enero.csv")

df_2020_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2020/flightlist_2020Febrero.csv")

df_2020_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2020/flightlist_2020Marzo.csv")

df_2020 = df_2020_ene.union(df_2020_feb).union(df_2020_mar)

In [6]:
df_2021_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2021/flightlist_2021Enero.csv")

df_2021_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2021/flightlist_2021Febrero.csv")

df_2021_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2021/flightlist_2021Marzo.csv")

df_2021 = df_2021_ene.union(df_2021_feb).union(df_2021_mar)

In [7]:
df_2022_ene = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2022/flightlist_2022Enero.csv")

df_2022_feb = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2022/flightlist_2022Febrero.csv")

df_2022_mar = spark.read.option("header", True).option("inferSchema", False) \
    .csv("C:/Py/HolaMundo/data/2022/flightlist_2022Marzo.csv")

df_2022 = df_2022_ene.union(df_2022_feb).union(df_2022_mar)

In [8]:
#Union de dataset
df = df_2019.union(df_2020).union(df_2021).union(df_2022)

In [ ]:
## Muestra de los datos
df.show(5)
df.printSchema()
print("Total de filas:", df.count())

+--------+------+------+------------+--------+------+-----------+--------------------+--------------------+--------------------+------------------+------------------+----------+------------------+------------------+----------+
|callsign|number|icao24|registration|typecode|origin|destination|           firstseen|            lastseen|                 day|        latitude_1|       longitude_1|altitude_1|        latitude_2|       longitude_2|altitude_2|
+--------+------+------+------------+--------+------+-----------+--------------------+--------------------+--------------------+------------------+------------------+----------+------------------+------------------+----------+
|   HVN19|  NULL|888152|        NULL|    NULL|  YMML|       LFPG|2018-12-31 00:43:...|2019-01-01 04:56:...|2019-01-01 00:00:...|-37.65948486328125|144.80442128282908|     304.8| 48.99531555175781| 2.610802283653846|    -53.34|
|  CCA839|  NULL|780ad1|        NULL|    NULL|  YMML|       LEBL|2018-12-31 00:53:...|2019-0

## Descripción de variables

El conjunto de datos contiene información general de vuelos, incluyendo identificadores de aeronaves, aeropuertos de origen y destino, tiempos de observación y coordenadas geográficas.

Las variables utilizadas en este análisis fueron principalmente:

- `origin`: aeropuerto de origen
- `destination`: aeropuerto de destino
- `typecode`: tipo de aeronave
- `firstseen`: fecha y hora inicial observada
- `lastseen`: fecha y hora final observada
- `latitude_1`, `longitude_1`: coordenadas iniciales
- `latitude_2`, `longitude_2`: coordenadas finales

## Limpieza y preparación de datos

Todas las columnas quedaron en formato texto.  
Por ello, fue necesario limpiar valores nulos y convertir las variables requeridas a tipos numéricos antes de aplicar el modelo.

In [10]:
df = df.dropna(subset=[
    "origin",
    "destination",
    "typecode",
    "firstseen",
    "lastseen",
    "latitude_1",
    "longitude_1",
    "latitude_2",
    "longitude_2"
])

In [11]:
df = df.withColumn("latitude_1", col("latitude_1").cast("double")) \
       .withColumn("longitude_1", col("longitude_1").cast("double")) \
       .withColumn("latitude_2", col("latitude_2").cast("double")) \
       .withColumn("longitude_2", col("longitude_2").cast("double"))

In [12]:
##Conversion de fechas 
df = df.withColumn("firstseen_ts", unix_timestamp(col("firstseen"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("lastseen_ts", unix_timestamp(col("lastseen"), "yyyy-MM-dd HH:mm:ss"))

In [15]:
## Conversion de fechas 
from pyspark.sql.functions import col, substring, unix_timestamp, round

df = df.withColumn("firstseen_clean", substring(col("firstseen"), 1, 19)) \
       .withColumn("lastseen_clean", substring(col("lastseen"), 1, 19))

df = df.withColumn("firstseen_ts", unix_timestamp(col("firstseen_clean"), "yyyy-MM-dd HH:mm:ss")) \
       .withColumn("lastseen_ts", unix_timestamp(col("lastseen_clean"), "yyyy-MM-dd HH:mm:ss"))

In [16]:
df.select(
    "firstseen", "firstseen_clean", "firstseen_ts",
    "lastseen", "lastseen_clean", "lastseen_ts"
).show(5, truncate=False)

+-------------------------+-------------------+------------+-------------------------+-------------------+-----------+
|firstseen                |firstseen_clean    |firstseen_ts|lastseen                 |lastseen_clean     |lastseen_ts|
+-------------------------+-------------------+------------+-------------------------+-------------------+-----------+
|2018-12-31 01:05:29+00:00|2018-12-31 01:05:29|1546239929  |2019-01-01 04:09:29+00:00|2019-01-01 04:09:29|1546337369 |
|2018-12-31 01:07:21+00:00|2018-12-31 01:07:21|1546240041  |2019-01-01 03:32:59+00:00|2019-01-01 03:32:59|1546335179 |
|2018-12-31 01:18:29+00:00|2018-12-31 01:18:29|1546240709  |2019-01-01 04:32:28+00:00|2019-01-01 04:32:28|1546338748 |
|2018-12-31 01:49:28+00:00|2018-12-31 01:49:28|1546242568  |2019-01-01 04:11:38+00:00|2019-01-01 04:11:38|1546337498 |
|2018-12-31 03:31:48+00:00|2018-12-31 03:31:48|1546248708  |2019-01-01 04:30:10+00:00|2019-01-01 04:30:10|1546338610 |
+-------------------------+-------------------+-

In [17]:
## Eliminamos nulos
df = df.dropna(subset=["firstseen_ts", "lastseen_ts"])

## Variable objetivo

Para este trabajo se definió como variable objetivo la duración estimada del vuelo en minutos, calculada a partir de la diferencia entre `lastseen` y `firstseen`.

In [18]:
from pyspark.sql.functions import round

df = df.withColumn(
    "duracion_min",
    round((col("lastseen_ts") - col("firstseen_ts")) / 60, 2)
)

In [19]:
df.select("firstseen", "lastseen", "firstseen_ts", "lastseen_ts").show(5, truncate=False)

+-------------------------+-------------------------+------------+-----------+
|firstseen                |lastseen                 |firstseen_ts|lastseen_ts|
+-------------------------+-------------------------+------------+-----------+
|2018-12-31 01:05:29+00:00|2019-01-01 04:09:29+00:00|1546239929  |1546337369 |
|2018-12-31 01:07:21+00:00|2019-01-01 03:32:59+00:00|1546240041  |1546335179 |
|2018-12-31 01:18:29+00:00|2019-01-01 04:32:28+00:00|1546240709  |1546338748 |
|2018-12-31 01:49:28+00:00|2019-01-01 04:11:38+00:00|1546242568  |1546337498 |
|2018-12-31 03:31:48+00:00|2019-01-01 04:30:10+00:00|1546248708  |1546338610 |
+-------------------------+-------------------------+------------+-----------+
only showing top 5 rows



In [20]:
df = df.filter(col("duracion_min") > 0)

In [21]:
df.select("origin", "destination", "duracion_min").show(5)

+------+-----------+------------+
|origin|destination|duracion_min|
+------+-----------+------------+
|  YSSY|       EDDF|      1624.0|
|  LEMD|       LEMD|     1585.63|
|  YSSY|       LFPG|     1633.98|
|  UUEE|       EDDF|     1582.17|
|  KLDJ|       LFPG|     1498.37|
+------+-----------+------------+
only showing top 5 rows



## Preparación de variables para MLlib

Las variables categóricas como aeropuerto de origen, destino y tipo de aeronave fueron transformadas a valores numéricos mediante `StringIndexer`.  
Posteriormente, todas las variables se integraron en un vector de características con `VectorAssembler`.

In [22]:
idx_origin = StringIndexer(inputCol="origin", outputCol="origin_idx", handleInvalid="skip")
idx_dest = StringIndexer(inputCol="destination", outputCol="destination_idx", handleInvalid="skip")
idx_type = StringIndexer(inputCol="typecode", outputCol="typecode_idx", handleInvalid="skip")

In [23]:
assembler = VectorAssembler(
    inputCols=[
        "origin_idx",
        "destination_idx",
        "typecode_idx",
        "latitude_1",
        "longitude_1",
        "latitude_2",
        "longitude_2"
    ],
    outputCol="features"
)

## Modelo utilizado

Se utilizó un modelo de regresión lineal de PySpark MLlib con el propósito de estimar la duración de los vuelos a partir de las variables seleccionadas.

In [24]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="duracion_min"
)

In [25]:
pipeline = Pipeline(stages=[
    idx_origin,
    idx_dest,
    idx_type,
    assembler,
    lr
])

In [26]:
#Separacion de entrenamiento y prueba 
train, test = df.randomSplit([0.8, 0.2], seed=42)

model = pipeline.fit(train)
predictions = model.transform(test)

## Evaluación del modelo

Para evaluar el desempeño del modelo se utilizaron las métricas RMSE y R²:

- **RMSE**: mide el error promedio de predicción
- **R²**: indica qué proporción de la variabilidad de la duración del vuelo es explicada por el modelo

In [27]:
rmse = RegressionEvaluator(
    labelCol="duracion_min",
    predictionCol="prediction",
    metricName="rmse"
).evaluate(predictions)

r2 = RegressionEvaluator(
    labelCol="duracion_min",
    predictionCol="prediction",
    metricName="r2"
).evaluate(predictions)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 139.98271656078575
R2: 0.042926588358351725


## Conclusion 

El modelo presentó un RMSE elevado y un R² bajo, lo que indica una baja capacidad predictiva. Esto se debe principalmente a la falta de variables relevantes que influyen en la duración del vuelo, como condiciones operativas, clima o velocidad real de la aeronave